# 04 — A/B Test Statistical Analysis

### **Day 4's question:** is the difference between Test and Control real, and is it big enough to act on?

-> Input is the client-level KPI table built in '03_B_client_level_kpi_analysis'

-> Randomisation was by 'client_id', so the client is the unit here, which means:

**one row, one observation, regardless of how many journeys that client made.**

### Two questions per KPI, in this order:

**1.** could the difference be only chance?

**2.** is it large enough for Vanguard to act on?

----


## 1. Loading the KPI tables

Two files:

- 'client_kpi_analysis.csv' — one row per randomised client (=50,500 rows)
- 'client_completion_time.csv' — one row per client who has a measurable completion time

#### They are merged on 'client_id'
#### Clients with no completion time keep a blank, which is the correct way to do: someone who never confirmed has no time to report.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

DATA = "../data/cleaned/"          

kpi  = pd.read_csv(DATA + "client_kpi_analysis.csv")
time = pd.read_csv(DATA + "client_completion_time.csv")

# Only two columns are taken from 'time': it also carries a Variation column, and merging both would produce Variation_x / Variation_y
# how="left" keeps all 50,500 clients and attaches a time where one exists (and put NaN when it doesnt)
# An inner join would drop everyone who never completed and push the completion rate to 100%

kpi = kpi.merge(
    time[["client_id", "completion_minutes"]],
    on="client_id",
    how="left",
)

print("clients:", len(kpi))
print("with a completion time:", kpi["completion_minutes"].notna().sum())
kpi.head()

clients: 50500
with a completion time: 32791


,client_id,Variation,completed,has_start,not_completed,step_back_count,completion_minutes
0,9988021,Test,False,True,True,2,NaN
1,8320017,Test,True,True,False,0,1.983333
2,4033851,Control,True,True,False,3,24.866667
3,1982004,Test,True,True,False,0,3.000000
4,9294070,Control,False,True,True,0,NaN


### Checking the table before we use it

Four things we want to be sure of: 
- one row per client
- 50,500 of them
- nobody in both groups
- and no missing group labels

**If any of these is off, none of the tests below mean anything.**

In [2]:
print("rows:", len(kpi))
print("unique clients:", kpi["client_id"].nunique())
print("one row per client:", len(kpi) == kpi["client_id"].nunique())
print("clients in both groups:", (kpi.groupby("client_id")["Variation"].nunique() > 1).sum())
print("missing group:", kpi["Variation"].isna().sum())
print()
print(kpi["Variation"].value_counts())

rows: 50500
unique clients: 50500
one row per client: True
clients in both groups: 0
missing group: 0

Variation
Test       26968
Control    23532
Name: count, dtype: int64


### Do the rates match what 03_B reported?

-> 03_B reports 65.59% for Control and 69.29% for Test. 

-> Recomputing them from the CSV is how we know we're working with the same file those numbers came from, and not an older version of it.

In [3]:
check = kpi.groupby("Variation")["completed"].agg(["count", "sum", "mean"])
check["rate_pct"] = check["mean"] * 100
check.round(4)

# count = how many clients are in the variation groups
# sum = how many completed the process = reached 'confirm'

,count,sum,mean,rate_pct
Variation,,,,
Control,23532,15434,0.6559,65.5873
Test,26968,18687,0.6929,69.2932


-----

## 2. Primary KPI and business threshold

Set before looking at any result.

### Primary KPI: Completion Rate, at client level

It's the closest thing in the data to what Vanguard actually asked —> ***do more customers get to the end?***
The recommendation is built on this one.

Everything else in this notebook is secondary, and is there to explain *why* the primary KPI moved rather than to decide anything on its own.

### Denominator: all 50,500 randomised clients

Including the ones with no observed start. Agreed between the team, and the reasoning is written up in the decisions log.

### Threshold: 2 percentage points

Below that, we wouldn't recommend rolling out.

The measurement has a precision of its own. With about 25,000 clients in each group, the standard error on a difference in completion rates works out at roughly 0.42 pp, which puts the 95% margin at around ±0.82 pp.

->> Anything below that and we'd be setting a bar the data can't actually see. Two points is safely above that, and still small enough to be a target a business would recognise.

### Why fix it now instead of later

Picked after seeing the result, the threshold would be whatever made the answer come out the way we already wanted.

In [5]:
PRIMARY_KPI = "completed"
THRESHOLD_PP = 2.0     # percentage points on Completion Rate
ALPHA = 0.05

In [6]:
n = kpi["Variation"].value_counts()
n_t, n_c = n["Test"], n["Control"]
p = kpi["completed"].mean()

se = (p * (1 - p) / n_t + p * (1 - p) / n_c) ** 0.5
margin = 1.96 * se

print(f"overall completion rate: {p:.3f}")
print(f"standard error of the difference: {se*100:.2f} pp")
print(f"95% margin: ±{margin*100:.2f} pp")
print(f"threshold chosen: {THRESHOLD_PP} pp")

overall completion rate: 0.676
standard error of the difference: 0.42 pp
95% margin: ±0.82 pp
threshold chosen: 2.0 pp


-----

## 3. Hypotheses

For each KPI:

- **H0** — the Test and Control distributions are the same
- **H1** — they differ

Both tests are two-sided -> a redesign can just as easily make things worse, and we want to be able to see that if it happens!!

#### We only test the completion rate, not the noncompletion rate. The two always add up to 100%, so they're the same fact stated twice:
-> Control is 65.59% / 34.41% ; Test is 69.29% / 30.71%. 

-> Testing both would give the same p-value twice and make the multiple-comparison correction more difficult for no reason.

-> Noncompletion is reported as a descriptive figure only.

------

## 4. What the numbers look like

The ***raw comparison first***, before any testing. 

The last column is the plain difference between the two groups, so we can see what we're dealing with.

In [7]:
test = kpi[kpi["Variation"] == "Test"]
control = kpi[kpi["Variation"] == "Control"]

summary = pd.DataFrame({
    "control": [
        control["completed"].mean() * 100,
        (control["step_back_count"] > 0).mean() * 100,
        control["step_back_count"].mean(),
        control["completion_minutes"].median(),
    ],
    "test": [
        test["completed"].mean() * 100,
        (test["step_back_count"] > 0).mean() * 100,
        test["step_back_count"].mean(),
        test["completion_minutes"].median(),  # bc of the outliers, that pushed the mean, so better to use median in this case
    ],
}, index=[
    "completion_rate_pct",
    "step_back_rate_pct",
    "step_backs_mean",
    "completion_time_median_min",
])

summary["difference"] = summary["test"] - summary["control"]
summary.round(3)

,control,test,difference
completion_rate_pct,65.587,69.293,3.706
step_back_rate_pct,26.101,33.414,7.313
step_backs_mean,0.407,0.602,0.195
completion_time_median_min,4.517,3.950,-0.567


### **One result doesn't fit the rest**

-> The step back gap is +7.3 pp, almost double the completion gap of +3.7
->> It means the new site gets more people to the end, and gets them there faster, but more of them are moving backwards on the way.

#### We can read that two ways:
1- Either the design confuses people and they have to go back, OR
2- going back got easier and people now do it instead of giving up. 

-> In our opinion, the second one can fit better with more people completing BECAUSE:
->> ***if customers were lost we'd expect more abandonment, not less.*** 

-> But we can't actually tell from this data, so we're presenting it as an open question rather than picking the reading that suits us best.

#### Worth noting too: the step back numbers aren't adjusted for how many journeys each client made. If Test clients made more attempts, part of that 7.3 pp is just more chances to go backwards. That's in the limitations.

-----

**Note**:
Funnel conversion isn't repeated here, bc it's already in '03_customer_journey_analysis', at journey level, which is where it belongs.

-> The funnel is asking which screen loses people, and that happens inside a single attempt.

-> A client who stops at step_2 twice and then completes on the 3rd try shows up at client level as having reached confirm, and those two drop-offs at step_2 disappear. That's the one thing the funnel exists to find.

-> Journey level works here because we're just describing where people drop out. We're not running a test on it, and the independence problem only matters when there's a p-value involved.

------

## 5. Assumptions

-> The two rates are proportions, so there's nothing to check beyond having enough observations, and with 50,500 clients we do.

-> The times and the step back counts are a different matter. A **t-test assumes the data is roughly normal**, and if it isn't, the test isn't valid. 

-> So we look at the shape first and pick the test after.

In [8]:
numeric_kpis = ["completion_minutes", "step_back_count"]

pd.DataFrame({
    "skew":     kpi[numeric_kpis].skew(),
    "kurtosis": kpi[numeric_kpis].kurtosis(),
}).round(2)

,skew,kurtosis
completion_minutes,7.90,178.20
step_back_count,4.09,32.71


#### Both are strongly right-skewed. 
-> A skew of 0 means a symmetric bell, and anything around 1 already looks lopsided

-> Completion time comes out near 8 and that makes sense, because:
**three quarters of clients finish in under 7 minutes but the slowest one took 300**

-> We can see the same thing in the mean and the median->6.4 against 4.2

-> The long tail drags the mean up, so we use the median instead.

-> Step backs are skewed for a different reason. 35,347 of the 50,500 clients have zero, and one client has 25 step backs.

#### Kurtosis tells us how much of the spread is coming from extreme values. 
-> A normal distribution gives 0, and both of ours are well above that, so the outliers here aren't just a few odd cases.


#### Decision:
That rules out a t-test, since it assumes the data is roughly normal. We use Mann-Whitney for these two, and two-proportion z-tests for the rates.

-----

## 6. Tests

One row per KPI: the test used and the p-value.

In [9]:
def test_proportion(success_test, n_test_, success_ctrl, n_ctrl):
    """Two-proportion z-test. Returns z and the two-sided p-value."""
    p_pool = (success_test + success_ctrl) / (n_test_ + n_ctrl)
    se = (p_pool * (1 - p_pool) * (1 / n_test_ + 1 / n_ctrl)) ** 0.5
    z = (success_test / n_test_ - success_ctrl / n_ctrl) / se
    return z, 2 * (1 - stats.norm.cdf(abs(z)))


results = []

# --- proportions -------------------------------------------------
z, p = test_proportion(test["completed"].sum(), len(test),
                       control["completed"].sum(), len(control))
results.append({"kpi": "completion_rate", "test": "two-proportion z",
                "statistic": z, "p_value": p})

z, p = test_proportion((test["step_back_count"] > 0).sum(), len(test),
                       (control["step_back_count"] > 0).sum(), len(control))
results.append({"kpi": "step_back_rate", "test": "two-proportion z",
                "statistic": z, "p_value": p})

# --- distributions -----------------------------------------------
for name in ["completion_minutes", "step_back_count"]:
    stat, p = stats.mannwhitneyu(test[name].dropna(), control[name].dropna(),
                                 alternative="two-sided")
    results.append({"kpi": name, "test": "Mann-Whitney",
                    "statistic": stat, "p_value": p})

results = pd.DataFrame(results).set_index("kpi")
results

,test,statistic,p_value
kpi,,,
completion_rate,two-proportion z,8.874514e+00,0.000000e+00
step_back_rate,two-proportion z,1.788843e+01,0.000000e+00
completion_minutes,Mann-Whitney,1.207712e+08,9.750612e-49
step_back_count,Mann-Whitney,3.424035e+08,6.658591e-81


#### All four KPIs come out significant, which is the expectation for a sample of 50,500 clients

-> bc at this sample size almost any difference will show up as significant, so these p-values mostly tell us the differences aren't random noise -> ***NOT that they matter***.

-> The two zeros aren't really zero. The p-value is just smaller than Python can represent (anything below about 1e-308 prints as 0), so what it means is "extremely small", not "impossible".

-> The useful question is: **how big are these differences, and does the completion rate clear the 2 pp threshold we set?** 

-> Also: these p-values haven't been corrected for running four tests yet. That's in the "multiple comparison" section

------

## 7. Effect sizes and confidence intervals

-> A **p-value says whether a difference is detectable**. It says nothing about how big it is

-> The **confidence interval** below gives us a range in percentage points and minutes, **which is what Vanguard would need** in order to decide whether to roll the new site out.

In [10]:
def proportion_diff_ci(success_test, n_test_, success_ctrl, n_ctrl, conf=0.95):
    """95% CI for the difference between two proportions, in percentage points."""
    p1, p2 = success_test / n_test_, success_ctrl / n_ctrl
    se = (p1 * (1 - p1) / n_test_ + p2 * (1 - p2) / n_ctrl) ** 0.5
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    diff = p1 - p2
    return diff * 100, (diff - z * se) * 100, (diff + z * se) * 100


diff, low, high = proportion_diff_ci(
    test["completed"].sum(), len(test),
    control["completed"].sum(), len(control),
)

print(f"Completion Rate difference: {diff:+.2f} pp")
print(f"95% CI: [{low:+.2f}, {high:+.2f}] pp")
print(f"Business threshold: {THRESHOLD_PP} pp")

Completion Rate difference: +3.71 pp
95% CI: [+2.89, +4.53] pp
Business threshold: 2.0 pp


In [11]:
sb_diff, sb_low, sb_high = proportion_diff_ci(
    (test["step_back_count"] > 0).sum(), len(test),
    (control["step_back_count"] > 0).sum(), len(control),
)

print(f"Step Back Rate difference: {sb_diff:+.2f} pp")
print(f"95% CI: [{sb_low:+.2f}, {sb_high:+.2f}] pp")

Step Back Rate difference: +7.31 pp
95% CI: [+6.52, +8.11] pp


#### Same function, applied to the step back rate. 
-> The only difference is '> 0', which turns the count into a yes/no ("did this client ever go back?") so we can treat it as a proportion.

-> The interval sits between +6.52 and +8.11 -> there's **no doubt ***Test clients*** go back more often**. 

-> It's also almost double the completion gap: +7.31 against +3.71 -> the KPI pointing the wrong way is the largest effect we have.

##### **HOWEVER careful with the sign here, bc positive only means Test has *more* of something: more completions is good, more step backs isn't. Also the median time came out negative, which is also good, because less time is better.**

-> to be fair, a point of completion rate isn't worth the same to Vanguard as a point of step backs, so we're not comparing like with like. but it's still the biggest movement we found, and it's the one going the wrong way.

-> No threshold is printed here on purpose. Only the completion rate has a bar to clear; the step back rate is there to explain, not to decide

-----

### For completion time we bootstrap the difference in medians instead.
-> There's a formula for the standard error of a mean, but not a simple one for the median of a skewed distribution, so we simulate it.

-> The idea: treat our sample as if it were the whole population. We draw a new sample of the same size from it, with replacement, so some clients get picked twice and others not at all.

-> Each fake sample gives a slightly different median difference. We do it a thousand times, then take the 2.5th and 97.5th percentiles of those thousand results -> that's our 95% interval.

-> The 'seed=42' is there so anyone re-running this gets the same interval we did.

In [12]:
def bootstrap_median_diff(a, b, n_boot=1000, seed=42):
    """95% CI for the difference in medians (test - control)."""
    r = np.random.default_rng(seed)
    a, b = a.dropna().to_numpy(), b.dropna().to_numpy()
    diffs = np.empty(n_boot)
    
    for i in range(n_boot):
        diffs[i] = (np.median(r.choice(a, len(a), replace=True))
                    - np.median(r.choice(b, len(b), replace=True)))
    return np.median(a) - np.median(b), np.percentile(diffs, [2.5, 97.5])


d, (lo, hi) = bootstrap_median_diff(test["completion_minutes"], control["completion_minutes"])
print(f"Median completion time: {d:+.2f} min   95% CI [{lo:+.2f}, {hi:+.2f}]")

Median completion time: -0.57 min   95% CI [-0.65, -0.48]


### How much faster is it, really?

-> Test clients are about 34 seconds faster (0.57 min × 60), and the interval [-0.65, -0.48] is entirely below zero, so the direction isn't in doubt.

-> 34 seconds sounds small, but the Control median is 4.52 minutes, so 0.57 / 4.52 = roughly **13% faster**. That's the version worth putting in front of Vanguard.

-> Note the point estimate (-0.57) comes from the real data. Only the interval comes from the simulation -> the bootstrap measures uncertainty, it doesn't estimate the value.

-----

## 8. Multiple comparisons

-> We ran four tests at the 5% level. The chance that at least one of them comes out significant purely by luck is 1 - 0.95⁴ = about 19%, so roughly one in five. Holm corrects for that.

-> Our primary KPI was declared before we looked at anything, so the recommendation doesn't depend on this correction. Reporting it is what stops us from over-reading the secondary KPIs.

-> One thing to note: 'step_back_rate' and 'step_back_count' are measuring the same behaviour, one as incidence and one as intensity. They aren't really independent tests, and Holm treats them as if they were, so the correction here is a bit stricter than it needs to be.

In [13]:
def holm(p_values):
    """Holm-Bonferroni adjusted p-values."""
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    n = len(p)
    adjusted = np.empty(n)
    running_max = 0.0
    for rank, idx in enumerate(order):
        value = (n - rank) * p[idx]
        running_max = max(running_max, value)
        adjusted[idx] = min(running_max, 1.0)
    return adjusted


results["p_adjusted"] = holm(results["p_value"])
results["significant"] = results["p_adjusted"] < ALPHA
results

,test,statistic,p_value,p_adjusted,significant
kpi,,,,,
completion_rate,two-proportion z,8.874514e+00,0.000000e+00,0.000000e+00,True
step_back_rate,two-proportion z,1.788843e+01,0.000000e+00,0.000000e+00,True
completion_minutes,Mann-Whitney,1.207712e+08,9.750612e-49,9.750612e-49,True
step_back_count,Mann-Whitney,3.424035e+08,6.658591e-81,1.331718e-80,True


### CONCLUSION: Nothing changes, but it still worth saying

-> All four are still significant after the correction.

-> You can actually see Holm working in the table. It sorts the p-values smallest first, so the two zeros get multiplied by 4 and 3, +step_back_count' (the third smallest) by 2 -> 6.66e-81 becomes 1.33e-80, and 'completion_minutes' is the largest so it gets multiplied by 1 and doesn't move at all.

-> With p-values this small the correction doesn't really do anything. Multiplying 1e-49 by 4 still leaves an absurdly small number. We're doing it because it's the right procedure and because with four tests the risk of a false alarm is real, not because it was ever going to change our answer here

-> What this table does *not* say is whether any of these differences are big enough to matter. That's the next section.

-----

## 9. Does it matter?

This is what the threshold was for. 

The tests told us the differences are real; this tells us whether the main one is big enough to act on.

In [14]:
print("Primary KPI: Completion Rate (client level, all randomised clients)")
print(f"Observed difference: {diff:+.2f} pp   (95% CI {low:+.2f} to {high:+.2f})")
print(f"Threshold for rollout: {THRESHOLD_PP} pp")
print()

if low > THRESHOLD_PP:
    verdict = "the entire confidence interval is above the threshold"
elif diff > THRESHOLD_PP:
    verdict = "the estimate is above the threshold but the interval includes values below it"
elif high > THRESHOLD_PP:
    verdict = "the estimate is below the threshold, though the interval does not rule it out"
else:
    verdict = "the entire confidence interval is below the threshold"

print("Verdict:", verdict)

Primary KPI: Completion Rate (client level, all randomised clients)
Observed difference: +3.71 pp   (95% CI +2.89 to +4.53)
Threshold for rollout: 2.0 pp

Verdict: the entire confidence interval is above the threshold


#### Does +3.71 pp justify the rollout?

-> Yes, because the whole interval is way above 2 pp, not just the estimate.

-> That matters. If we only looked at 3.71 being bigger than 2, we'd be ignoring the uncertainty. The lower bound is 2.89, so even if the real effect turns out to be at the pessimistic end, we're still above the threshold we set.

-> So on the **completion rate the answer is yes**. 

##### -> But the step backs went up, and that's the larger of the two effects, so the recommendation can't just report the half that looks good.

-----

## 10. Data quality check: confirmations without a start

-> 03_B found 424 clients with no 'start' event, however 391 of those 424 reached 'confirm' anyway.

-> That changes what the "missing starts" mean. These aren't customers we lost before the first screen, it makes no sense -> we think they're completed processes where the first event just wasn't recorded. **It's a tracking problem, not a behaviour one.**

-> It's also not symmetric between the groups, which is why it's worth a section instead of a footnote.

In [15]:
no_start = kpi[~kpi["has_start"]]

quality = pd.DataFrame({
    "clients":        kpi.groupby("Variation").size(),
    "no_start":       no_start.groupby("Variation").size(),
    "no_start_but_completed":
        no_start[no_start["completed"]].groupby("Variation").size(),
})

quality["no_start_pct"] = quality["no_start"] / quality["clients"] * 100
quality.round(2)

,clients,no_start,no_start_but_completed,no_start_pct
Variation,,,,
Control,23532,135,116,0.57
Test,26968,289,275,1.07


In [16]:
z, p = test_proportion(
    quality.loc["Test", "no_start"],    quality.loc["Test", "clients"],
    quality.loc["Control", "no_start"], quality.loc["Control", "clients"],
)

print(f"Missing-start rate, Test vs Control:  z = {z:.2f}   p = {p:.2e}")

Missing-start rate, Test vs Control:  z = 6.12   p = 9.49e-10


-> The Test platform fails to record a 'start' event about twice as often as Control (1.07% vs 0.57%), and z = 6.12 means that gap isn't chance.

-> Most of these clients did complete, so the effect on the completion rate is small. But it's a defect in the new platform's event tracking, and someone at Vanguard who owns the instrumentation should know about it.

-> It also means we can't fully tell "no start recorded" apart from "never started". So we checked what happens if we exclude them.

In [17]:
# Sensitivity: what happens to the primary KPI if the 424 are excluded?
started = kpi[kpi["has_start"]]

s_test    = started[started["Variation"] == "Test"]
s_control = started[started["Variation"] == "Control"]

s_diff, s_low, s_high = proportion_diff_ci(
    s_test["completed"].sum(), len(s_test),
    s_control["completed"].sum(), len(s_control),
)

print(f"All randomised clients:      {diff:+.2f} pp   [{low:+.2f}, {high:+.2f}]")
print(f"Restricted to those who started: {s_diff:+.2f} pp   [{s_low:+.2f}, {s_high:+.2f}]")

All randomised clients:      +3.71 pp   [+2.89, +4.53]
Restricted to those who started: +3.54 pp   [+2.72, +4.37]


#### Would excluding them have changed the answer?

-> No. +3.54 pp instead of +3.71, and the interval still sits well above 2 pp. The conclusion holds either way.

-> Worth noticing which direction it moves: excluding the 424 makes the gap *smaller*, not bigger. So keeping them in wasn't the choice that flattered the Test group — it was the more cautious one.

------

## 11. Limitations

1. **Effective time is not available at client level.** It exists in the journey-level notebook
but wasn't carried into the client table, so all we have here is the total time from start
to confirm. *****A client who left the form open and came back later has that idle time counted
as time spent*****.

2. **Step backs are not adjusted for exposure.** A client who made four journeys had four times as many
opportunities to navigate backwards as one who made a single journey. If the groups differ in
journeys per client, part of the step-back gap reflects that rather than the interface. Journeys per
client averages 1.28 overall, so the effect is limited, but it is not zero.

3. **No error rate.** The event log records only the five process steps, with no explicit error event,
so no error KPI was constructed. *****Step backs are the closest available signal of navigation friction*****.
Repeated steps -> the same screen twice in a row —> would be a separate and arguably better signal, and
were not measured.

4. **Completion time is conditional on completing.** Only clients who reached 'confirm' have a time.*****!!***** If
the redesign changed *who* completes, it also changed who is in this comparison, and the two groups
being compared on time are not the same populations. This is a standard limitation of any
post-treatment measure and cannot be fixed by choosing a different test.

5. **Multiple attempts are collapsed.** A client who tried three times and succeeded on the third counts
as completed, the same as one who succeeded immediately. That is the correct choice for the business
question, but it hides effort.

-----

## Conclusion: should Vanguard implement the redesign?

**Yes. There's one screen we'd want looked at, and we now know which one.**

### The evidence

-> *At client level the completion rate is 69.29% for Test and 65.59% for Control*. 

-> ***That's a difference of +3.71 pp, with a 95% interval running from +2.89 to +4.53***. 

-> We set the threshold at 2 pp before running anything, and the whole interval is well above it, so this isn't a result that depends on us being just lucky.

> We also checked whether the denominator decision mattered. Restricting to clients with an observed start gives +3.54 pp [+2.72, +4.37], slightly smaller. Both intervals sit entirely above 2 pp, so **we'd recommend the rollout either way**.


### The secondary KPIs

-> *Test* clients complete *faster* too.

-> With a median of 3.95 minutes against 4.52, so around 34 seconds, or about 13%. The interval is [-0.65, -0.48], all of it below zero.

-> Then about the **step backs**, we can observe it goes the other way -> 33.41% of Test clients navigated backwards at least once, against 26.10% of Control. 

> ***That's +7.31 pp, and it's the largest effect anywhere in this analysis.***


### The step backs: two readings

-> We can interpret the step backs two ways, and on its own the client-level number won't decide for us.

1. Maybe the new interface confuses people, so they have to go back.
2. Or maybe going back got easy enough that people do it now instead of closing the tab.

-> The funnel breakdown in 03_B narrows it (see 03_B for the full table). **The extra backward navigation isn't spread evenly, it's concentrated at the front of the process.**

| backward move | Control per 100 clients | Test per 100 clients | difference |
| --- | ---: | ---: | ---: |
| step_1 → start | 10.58 | 23.75 | +13.17 |
| step_2 → step_1 | 5.91 | 11.65 | +5.74 |
| step_3 → step_2 | 10.05 | 8.46 | −1.59 |

*Interpretation:*
-> Test clients go back from step_1 to start more than twice as often as Control. Step 2 to step 1 is also roughly double. But step 3 to step 2 is actually *lower* in Test.

-> So the friction didn't get worse everywhere:
->> In Control it's spread across the funnel; **in Test one single transition accounts for nearly 40% of all backward moves.**

-> That leans towards the first reading, at least for the early steps. If people were just using an easier back button, we'd expect it spread out.
->> However, **concentrated on one transition looks more like something specific about getting from start to step_1.**

> Both readings can be true at the same time. One early screen that makes people backtrack, on a site that's easier to get through overall.
> That would explain **why completion went up and time went down while backward navigation went up at the same time**.


### What we'd do about it

-> The move between start and step_1 is where to look. 23.75 backward events per 100 Test clients against 10.58 for Control —> the biggest single difference we found.

-> This isn't a reason to hold the rollout. Test still completes more, and faster, and the completion rate clears the threshold comfortably. But that one screen is where the redesign is sending people backwards, and it's the obvious place to start the next round of design work.

-> Overall step back events per 100 clients: 40.7 for Control, 60.2 for Test. So there are about 48% more of them, and most of the excess sits in one place.

### Worth flagging separately

-> **The new platform fails to record a 'start' event about twice as often as the old one** -> 1.07% of Test clients against 0.57% of Control, z = 6.12

> It doesn't change our results much, bc nearly all of those clients completed anyway. But it's a bug in the new platform's event logging, and whoever owns that should be aware of it.


### One thing we checked, and one to keep in mind

-> We checked how 'backward_navigation' was built in 01: 'previous_step' is *computed with a groupby on 'client_id' and 'visit_id'*, so it **only ever compares steps inside the same session**. ->> The first event of each visit has no previous step and is never flagged. 
> That means none of these backward moves are an artefact of one session ending and another beginning —> they all happened inside a single sitting.

-> Which makes the 'step_3 → start' number even more interesting, because 8.29 per 100 Test clients against 4.53 for Control, all of it within one session
> That means people abandoning their progress and going back to the beginning while still in the process.

-> Some backward moves start from 'confirm', meaning the client had already finished. Control has 2.89 of those per 100 clients, Test has 1.13. Those are probably second applications rather than friction. We left them in, and it's worth saying that taking them out would make Test look slightly worse, not better.


### What we can't tell you

The limitations section above, in short:

- we're measuring total elapsed time rather than time actually spent working through the form
- the step back numbers don't account for how many attempts each client made
- there's no error rate, because nothing in the log records an error
- completion time only exists for clients who completed, so the two groups we compare on time aren't the same population

> None of this changes the completion rate result. It does mean the completion time and the two step back numbers are less solid, and we'd treat them as indications of what's happening rather than as measurements of it.


### Significance isn't the reason we're recommending this

-> Everything in this notebook came out significant. 

-> With 50,500 clients that was always going to happen, and on its own it doesn't mean much. 

-> Back in '02b' notebook, a baseline gap of seven hundredths of a phone call reached p = 0.001. What makes the completion rate worth acting on is the size of it, 3.71 points, and the fact that even the bottom of the interval stays above a threshold we committed to before we'd seen a single result.
